In [ ]:
import pandas as pd

file_g034 = 'G034_predictions.csv'
file_g035 = 'G035_predictions.csv'  

In [ ]:
df_g034 = pd.read_csv(file_g034)
df_g035 = pd.read_csv(file_g035)

# common_cohorts = set(df_g034['cohort']).intersection(df_g035['cohort'])
# df_g034 = df_g034[df_g034['cohort'].isin(common_cohorts)]
# df_g035 = df_g035[df_g035['cohort'].isin(common_cohorts)]

In [ ]:
# common_cohorts

In [ ]:
merged = df_g034.merge(df_g035, on=['cohort', 'elapsed_months', 'current_month'], how='outer', suffixes=('_g034', '_g035'))


In [ ]:
merged.columns

In [ ]:
for col in merged.columns:
    if merged[col].dtype in ['int64', 'float64']:
        merged[col].fillna(0, inplace=True)
    elif merged[col].dtype == 'object':
        merged[col].fillna('', inplace=True)

In [ ]:
result = pd.DataFrame()
result['cohort'] = merged['cohort']
result['elapsed_months'] = merged['elapsed_months']
result['current_month'] = merged['current_month']
result['game_id'] = (merged['game_id_g034'].fillna('') + '+' +
                     merged['game_id_g035'].fillna(''))

In [ ]:
numerical_columns = ['absolute_revenue_actual', 'total_cac', 'absolute_revenue_bad',
                     'absolute_revenue_good', 'absolute_revenue_expected']

for col in numerical_columns:
    result[col] = (merged[f'{col}_g034'].fillna(0) +
                   merged[f'{col}_g035'].fillna(0))

result['total_cac'].replace(0, 1e-9, inplace=True)

result['transactional_actual'] = result['absolute_revenue_actual'] / result['total_cac']
result['transactional_bad'] = result['absolute_revenue_bad'] / result['total_cac']
result['transactional_good'] = result['absolute_revenue_good'] / result['total_cac']
result['transactional_expected'] = result['absolute_revenue_expected'] / result['total_cac']

In [ ]:
result['cumulative_actual'] = 0
result['cumulative_bad'] = 0
result['cumulative_good'] = 0
result['cumulative_expected'] = 0

In [ ]:
for cohort, group in result.groupby('cohort'):
    cumulative_actual = 0
    cumulative_bad = 0
    cumulative_good = 0
    cumulative_expected = 0

    for index, row in group.iterrows():
        cumulative_actual += row['transactional_actual']
        cumulative_bad += row['transactional_bad']
        cumulative_good += row['transactional_good']
        cumulative_expected += row['transactional_expected']

        result.loc[index, 'cumulative_actual'] = cumulative_actual
        result.loc[index, 'cumulative_bad'] = cumulative_bad
        result.loc[index, 'cumulative_good'] = cumulative_good
        result.loc[index, 'cumulative_expected'] = cumulative_expected

In [ ]:
result = result[df_g034.columns]

In [ ]:
output_file = 'G034_G035_combined.csv'
result.to_csv(output_file, index=False)

print(f'Results saved to {output_file}')